## TEMPO -- Initialization

**TEMPO** discovers $M$ latent regimes in trajectory data via Expectation-Maximization,
training a dedicated basis for each regime.

Initialization (3 steps):
1. **Global POD** -- SVD on all trajectories $\{s^{(n)}\}_{n=1}^N$ gives coefficients $\alpha^{(n)} \in \mathbb{R}^P$
2. **GMM** -- fit Gaussian mixture on $\{\alpha^{(n)}\}$ gives soft assignments $\gamma \in \mathbb{R}^{N \times M}$
3. **Regime bases** -- train $M$ weighted bases: $\min_{\Phi_m} \sum_n \gamma_{nm} \|s^{(n)} - \hat{s}_m^{(n)}\|^2$

In [ ]:
import os, sys, pathlib
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from sklearn.manifold import TSNE
from umap import UMAP

sys.path.insert(0, str(pathlib.Path("..").resolve()))
from models.tempo import TEMPOTrainer, TEMPOConfig, pod_factory, fourier_pod_factory
from models.pod import PODConfig
from models.fourier_neural_pod import FourierNeuralPODConfig
from utils.datasets import load

if torch.backends.mps.is_available():
    DEVICE = 'mps'
elif torch.cuda.is_available():
    DEVICE = 'cuda'
else:
    DEVICE = 'cpu'

DATA_DIR = os.path.expanduser("~/data/1D/Burgers/Train")


def draw_cov_ellipse(ax, mean2d, cov2d, color, n_std=2.0, **kwargs):
    vals, vecs = np.linalg.eigh(cov2d)
    angle = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))
    w, h = 2 * n_std * np.sqrt(np.abs(vals))
    ax.add_patch(Ellipse(mean2d, w, h, angle=angle,
                         edgecolor=color, facecolor="none", lw=2, **kwargs))

### Data

In [ ]:
nu_values = [0.001, 0.01, 0.1, 1.0]
N_samples = 6000

In [ ]:
tensors, kappas = [], []

In [ ]:
for nu in nu_values:
    path = os.path.join(DATA_DIR, f"1D_Burgers_Sols_Nu{nu}.hdf5")
    data = load(f"Burgers_Nu{nu}", path=path)
    u = data["tensor"][:N_samples]  # (N, Nt, Nx) from DaRUS
    if u.ndim == 4:
        u = u[..., 0]
    t_np = data["t-coordinate"]
    x_np = data["x-coordinate"]
    N, Nt, Nx = u.shape
    t_np = t_np[:Nt]  # clip to match tensor length
    tensors.append(u.reshape(N, -1))  # (N, Nt*Nx)
    kappas.append(np.full(N, nu))

In [ ]:
s_np = np.concatenate(tensors, axis=0)    # (N_total, Nt*Nx)
k_np_all = np.concatenate(kappas, axis=0)  # (N_total,)

# spatiotemporal coordinate grid: col 0 = x, col 1 = t
x_grid = torch.tensor(x_np, dtype=torch.float32)
t_grid = torch.tensor(t_np, dtype=torch.float32)
tt, xx = torch.meshgrid(t_grid, x_grid, indexing='ij')  # (Nt, Nx)
x_flat = torch.stack([xx.flatten(), tt.flatten()], dim=1)  # (Nt*Nx, 2)

s = torch.tensor(s_np, dtype=torch.float32)  # (N_total, Nt*Nx) on CPU
x = x_flat.to(DEVICE)  # (Nt*Nx, 2)
kappa = torch.tensor(k_np_all[:, None], dtype=torch.float32)  # (N_total, 1) on CPU

In [ ]:
print(f"s={s.shape}, x={x.shape}, kappa={kappa.shape}")
print(f"  {len(nu_values)} regimes × {s.shape[0]//len(nu_values)} trajectories, Nt*Nx={s.shape[1]}")

# **TEMPO:**

### **POD**

In [7]:
cfg = TEMPOConfig(
    M=3,
    basis_config=PODConfig(max_modes=1),
    basis_factory=pod_factory,
    P_global = 5
)

regime_colors = np.array([
    [0.27, 0.51, 0.71, 1.0],
    [0.84, 0.37, 0.30, 1.0],
    [0.30, 0.64, 0.42, 1.0],
    [0.75, 0.55, 0.24, 1.0],
])[:cfg.M]

In [ ]:
trainer = TEMPOTrainer(cfg)
trainer._initialize(s, x, t=None, kappa=kappa)

In [ ]:
print(f"alpha:  {trainer.alpha.shape}")   # (N, P_global)
print(f"gamma:  {trainer.gamma.shape}")   # (N, M)
print(f"pi:     {trainer.pi.shape}")      # (M,)
print(f"mu:     {trainer.mu.shape}")      # (M, P_global)
print(f"Sigma:  {trainer.Sigma.shape}")   # (M, P_global, P_global)
print(f"bases:  {len(trainer.trainers)}")

Latent structure of Burgers **trajectories** via global POD coefficients $\alpha \in \mathbb{R}^P$:

- **UMAP:**

In [ ]:
N_viz = min(5632, len(s))
idx = np.random.choice(len(s), N_viz, replace=False)

kappa_viz = kappa[idx, 0].numpy()
alpha_viz = trainer.alpha[idx].cpu().numpy()
embedding = UMAP(n_neighbors=30, min_dist=0.0).fit_transform(alpha_viz)

nu_unique = np.unique(kappa_viz)
tab_colors = plt.cm.tab10(np.linspace(0, 0.9, len(nu_unique)))
nu_to_col = {nu: c for nu, c in zip(nu_unique, tab_colors)}
point_colors = np.array([nu_to_col[float(nu)] for nu in kappa_viz])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(embedding[:, 0], embedding[:, 1], c=point_colors, s=3, alpha=0.5, rasterized=True)
axes[0].set_title(r"UMAP -- colored by $\nu$", fontweight="bold")
axes[0].set_xlabel("UMAP-1"); axes[0].set_ylabel("UMAP-2")
handles = [plt.scatter([], [], color=c, s=40, label=rf"$\nu={nu:.3f}$")
           for nu, c in zip(nu_unique, tab_colors)]
axes[0].legend(handles=handles, fontsize=9)

axes[1].scatter(alpha_viz[:, 0], alpha_viz[:, 1], c=point_colors, s=3, alpha=0.5, rasterized=True)
axes[1].set_title(r"POD space: $\alpha_1$ vs $\alpha_2$", fontweight="bold")
axes[1].set_xlabel(r"$\alpha_1$"); axes[1].set_ylabel(r"$\alpha_2$")
handles2 = [plt.scatter([], [], color=c, s=40, label=rf"$\nu={nu:.3f}$")
            for nu, c in zip(nu_unique, tab_colors)]
axes[1].legend(handles=handles2, fontsize=9)

for ax in axes:
    ax.grid(True, ls="--", alpha=0.2)
    ax.spines[["top", "right"]].set_visible(False)

plt.suptitle("Burgers trajectories -- latent structure", fontweight="bold", fontsize=14)
plt.tight_layout()
plt.show()

Regime assignment after GMM initialization:

In [ ]:
reducer = UMAP(n_neighbors=30, min_dist=0.0, random_state=39)
embedding = reducer.fit_transform(alpha_viz)

mu_np = trainer.mu.cpu().numpy()  # (M, P_global)
mu_umap = reducer.transform(mu_np)
regime_colors = plt.cm.Set1(np.linspace(0, 0.8, cfg.M))
hard_labels = trainer.gamma[idx].argmax(dim=1).cpu().numpy()

In [ ]:
nu_unique = np.unique(kappa_viz)
nu_palette = plt.cm.Set2(np.linspace(0, 0.8, len(nu_unique)))
nu_to_col = {nu: c for nu, c in zip(nu_unique, nu_palette)}
point_colors = np.array([nu_to_col[float(nu)] for nu in kappa_viz])

star_kw = dict(s=400, marker="*", zorder=10, edgecolors="white", linewidths=1.5)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(embedding[:, 0], embedding[:, 1],
                c=regime_colors[hard_labels], s=3, alpha=0.35, rasterized=True)
axes[0].scatter(mu_umap[:, 0], mu_umap[:, 1], c=regime_colors, **star_kw)
axes[0].set_title("UMAP -- GMM regime assignment", fontweight="bold")
handles = [plt.scatter([], [], color=regime_colors[m], s=50, label=f"Regime {m+1}")
           for m in range(cfg.M)]
axes[0].legend(handles=handles, fontsize=9)

axes[1].scatter(embedding[:, 0], embedding[:, 1],
                c=point_colors, s=3, alpha=0.3, rasterized=True)
axes[1].scatter(mu_umap[:, 0], mu_umap[:, 1], c=regime_colors, **star_kw)
axes[1].set_title(r"UMAP -- colored by $\nu$", fontweight="bold")
handles = [plt.scatter([], [], color=nu_to_col[nu], s=50,
                       label=rf"$\nu = {nu:.3f}$") for nu in nu_unique]
axes[1].legend(handles=handles, fontsize=9)

axes[2].scatter(alpha_viz[:, 0], alpha_viz[:, 1],
                c=point_colors, s=3, alpha=0.3, rasterized=True)
for m in range(cfg.M):
    cov2d = trainer.Sigma[m, :2, :2].cpu().numpy()
    mean2d = mu_np[m, :2]
    draw_cov_ellipse(axes[2], mean2d, cov2d, color=regime_colors[m], linestyle="--")
    axes[2].scatter(*mean2d, color=regime_colors[m], **star_kw, label=f"Regime {m+1}")
axes[2].set_title(r"POD space: $\alpha_1$ vs $\alpha_2$", fontweight="bold")
axes[2].set_xlabel(r"$\alpha_1$"); axes[2].set_ylabel(r"$\alpha_2$")
axes[2].legend(fontsize=9)

for ax in axes[:2]:
    ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
for ax in axes:
    ax.grid(True, ls="--", alpha=0.2)
    ax.spines[["top", "right"]].set_visible(False)

plt.suptitle("Burgers trajectories -- GMM regime structure", fontweight="bold", fontsize=14)
plt.tight_layout()
plt.show()

#### t-SNE

In [ ]:
alpha_with_mu = np.vstack([alpha_viz, mu_np])

tsne = TSNE(n_components=2, perplexity=50, random_state=42, n_jobs=-1)
emb_all = tsne.fit_transform(alpha_with_mu)

tsne_emb = emb_all[:-cfg.M]
mu_tsne = emb_all[-cfg.M:]

Python(29221) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


In [ ]:
N_tsne = min(5000, len(alpha_viz))
idx_tsne = np.random.choice(len(alpha_viz), N_tsne, replace=False)

alpha_tsne = alpha_viz[idx_tsne]
kappa_tsne = kappa_viz[idx_tsne]
hard_tsne = trainer.gamma[idx][idx_tsne].argmax(dim=1).cpu().numpy()
point_col_tsne = np.array([nu_to_col[float(nu)] for nu in kappa_tsne])

alpha_with_mu = np.vstack([alpha_tsne, mu_np])
emb_all = TSNE(n_components=2, perplexity=50, random_state=42, n_jobs=-1).fit_transform(alpha_with_mu)
tsne_emb = emb_all[:-cfg.M]
mu_tsne = emb_all[-cfg.M:]

star_kw = dict(s=400, marker="*", zorder=10, edgecolors="white", linewidths=1.5)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(tsne_emb[:, 0], tsne_emb[:, 1],
                c=regime_colors[hard_tsne], s=3, alpha=0.35, rasterized=True)
axes[0].scatter(mu_tsne[:, 0], mu_tsne[:, 1], c=regime_colors, **star_kw)
axes[0].set_title("t-SNE -- GMM regime assignment", fontweight="bold")
handles = [plt.scatter([], [], color=regime_colors[m], s=50, label=f"Regime {m+1}")
           for m in range(cfg.M)]
axes[0].legend(handles=handles, fontsize=9)

axes[1].scatter(tsne_emb[:, 0], tsne_emb[:, 1],
                c=point_col_tsne, s=3, alpha=0.35, rasterized=True)
axes[1].scatter(mu_tsne[:, 0], mu_tsne[:, 1], c=regime_colors, **star_kw)
axes[1].set_title(r"t-SNE colored by $\nu$", fontweight="bold")
handles = [plt.scatter([], [], color=nu_to_col[nu], s=50,
                       label=rf"$\nu = {nu:.3f}$") for nu in nu_unique]
axes[1].legend(handles=handles, fontsize=9)

for ax in axes:
    ax.set_xlabel("t-SNE-1"); ax.set_ylabel("t-SNE-2")
    ax.grid(True, ls="--", alpha=0.2)
    ax.spines[["top", "right"]].set_visible(False)

plt.suptitle("Burgers trajectories -- t-SNE latent structure", fontweight="bold", fontsize=14)
plt.tight_layout()
plt.show()

#### **Fourier Neural POD**

In [ ]:
cfg_f = TEMPOConfig(
    M=3,
    basis_config=FourierNeuralPODConfig(max_modes=1, n_epochs_mode=5, n_epochs_mean=5),
    basis_factory=fourier_pod_factory,
    P_global=3,
)
trainer_f = TEMPOTrainer(cfg_f)
trainer_f._initialize(s, x, t=None, kappa=kappa)

In [ ]:
N_viz_f = min(5632, len(s))
idx_f   = np.random.choice(len(s), N_viz_f, replace=False)

kappa_viz_f = kappa[idx_f, 0].numpy()
alpha_viz_f = trainer_f.alpha[idx_f].cpu().numpy()

In [ ]:
reducer_f = UMAP(n_neighbors=30, min_dist=0.0, random_state=39)
embedding_f = reducer_f.fit_transform(alpha_viz_f)

mu_np_f = trainer_f.mu.cpu().numpy()
mu_umap_f = reducer_f.transform(mu_np_f)
regime_col_f = plt.cm.Set1(np.linspace(0, 0.8, cfg_f.M))
hard_labels_f = trainer_f.gamma[idx_f].argmax(dim=1).cpu().numpy()

In [ ]:
nu_unique_f = np.unique(kappa_viz_f)
nu_pal_f = plt.cm.Set2(np.linspace(0, 0.8, len(nu_unique_f)))
nu_to_col_f = {nu: c for nu, c in zip(nu_unique_f, nu_pal_f)}
pt_col_f = np.array([nu_to_col_f[float(nu)] for nu in kappa_viz_f])

star_kw = dict(s=400, marker="*", zorder=10, edgecolors="white", linewidths=1.5)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(embedding_f[:, 0], embedding_f[:, 1],
                c=regime_col_f[hard_labels_f], s=3, alpha=0.35, rasterized=True)
axes[0].scatter(mu_umap_f[:, 0], mu_umap_f[:, 1], c=regime_col_f, **star_kw)
axes[0].set_title("UMAP -- GMM regime assignment", fontweight="bold")
handles = [plt.scatter([], [], color=regime_col_f[m], s=50, label=f"Regime {m+1}")
           for m in range(cfg_f.M)]
axes[0].legend(handles=handles, fontsize=9)

axes[1].scatter(embedding_f[:, 0], embedding_f[:, 1],
                c=pt_col_f, s=3, alpha=0.35, rasterized=True)
axes[1].scatter(mu_umap_f[:, 0], mu_umap_f[:, 1], c=regime_col_f, **star_kw)
axes[1].set_title(r"UMAP -- colored by $\nu$", fontweight="bold")
handles = [plt.scatter([], [], color=nu_to_col_f[nu], s=50,
                       label=rf"$\nu = {nu:.3f}$") for nu in nu_unique_f]
axes[1].legend(handles=handles, fontsize=9)

axes[2].scatter(alpha_viz_f[:, 0], alpha_viz_f[:, 1],
                c=pt_col_f, s=3, alpha=0.35, rasterized=True)
for m in range(cfg_f.M):
    cov2d = trainer_f.Sigma[m, :2, :2].cpu().numpy()
    mean2d = mu_np_f[m, :2]
    draw_cov_ellipse(axes[2], mean2d, cov2d, color=regime_col_f[m], linestyle="--")
    axes[2].scatter(*mean2d, color=regime_col_f[m], **star_kw, label=f"Regime {m+1}")
axes[2].set_title(r"POD space: $\alpha_1$ vs $\alpha_2$", fontweight="bold")
axes[2].set_xlabel(r"$\alpha_1$"); axes[2].set_ylabel(r"$\alpha_2$")
axes[2].legend(fontsize=9)

for ax in axes[:2]:
    ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
for ax in axes:
    ax.grid(True, ls="--", alpha=0.2)
    ax.spines[["top", "right"]].set_visible(False)

plt.suptitle("Burgers trajectories -- FourierNeuralPOD regime structure",
             fontweight="bold", fontsize=14)
plt.tight_layout()
plt.show()